In [1]:
import pickle
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from datetime import datetime
from training_utils import TorchTrainer2
import time
import torch

from velnet_utils import MyDS as DS
from velnet_utils import MyLoss as Loss
from velnet_utils import Net0317 as Net

import numpy as np
import os

def count_trainable(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

In [2]:

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# td_folder = '/zjbd/zd1/dafei/piv_training/jet160roi320'
# td_folder = '/zjbd/zd1/dafei/piv_training/jet160'
# td_folder = '/zjbd/zd1/dafei/piv_training/jet128r2'
# td_folder = '/zjbd/zd1/dafei/piv_training/jet256'
td_folder = '/zjbd/zd1/dafei/piv_training/jet320'


batch_size = 8
nt = 5

x_folder = os.path.join(td_folder, 'x')
y_folder = os.path.join(td_folder, 'y')
y_list = os.listdir(y_folder)
with open(os.path.join(td_folder, 'labels.pickle'), 'rb') as handle:
    labels = pickle.load(handle)
data_tags = labels['data_tags']

validation_tags = data_tags[2:3]

print(f'data tags: {data_tags}. test data tag: {validation_tags}')

training = [s for s in y_list if not any(p in s for p in validation_tags)]
validation = [s for s in y_list if any(p in s for p in validation_tags)]


print(f'y # :{len(y_list)}')
print(f'training: {len(training)}')
print(f'validation: {len(validation)}')

train_ds = DS(training, labels, x_folder, y_folder, nt)
validate_ds = DS(validation, labels, x_folder, y_folder, nt)
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=8, persistent_workers=True)
validate_dl = DataLoader(validate_ds, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=8, persistent_workers=True)


np.random.seed(66)
torch.manual_seed(88)

vol_dict = dict(vol_nc=64, vol_nz=100, vol_ds=(1, 4, 2), ds_zyx=(4, 16, 16))
cv_dict = dict(win=(19, 9, 19), cv_ds = (4, 4, 8))
dt1_dict = dict(use=True, step=1, md_zyx=(1, 2, 2), cv_nc=64)
dt2_dict = dict(use=True, step=2, md_zyx=(2, 3, 3), cv_nc=64)
net = Net(nt, vol_dict, cv_dict, dt1_dict, dt2_dict).to(device)


training_result_path = r'/home/dafei/piv/training_results/tp16_03-03_16-36'  # 100
# training_result_path = r'/home/dafei/piv/training_results/tp16_03-10_18-28'  # 200

checkpoint = torch.load(os.path.join(training_result_path, "net.pt"), map_location=device, weights_only=False)
pretrained_sd = checkpoint['state_dict']
new_sd = {f"layer1.{k}": v for k, v in pretrained_sd.items()}
net.load_state_dict(new_sd, strict=False)

# save
# checkpoints = None
save_dir = os.path.join(os.getcwd(), 'training_results')
time_now = datetime.today().strftime('%m-%d_%H-%M')
path_save = os.path.join(save_dir, os.path.basename(td_folder)+'_'+time_now)
os.makedirs(path_save, exist_ok=True)
print(f'training result folder: {path_save}')
checkpoints = dict(file_name=os.path.join(path_save, 'net.pt'),  # state
                   note=''
                   )


# loss
my_loss2 = Loss(lam_w=1.0)
# optimizer
for p in net.layer1.parameters():
    p.requires_grad = False
for p in net.layer2.parameters():
    p.requires_grad = True
print("layer 2 trainable:", count_trainable(net))

optimizer2 = torch.optim.AdamW(net.layer2.parameters(), lr=5e-4, weight_decay=1e-3)
scheduler2 = ReduceLROnPlateau(optimizer2, factor=0.1, patience=1, threshold=0.004, min_lr=1e-6)
trainer2 = TorchTrainer2(net, my_loss2, optimizer2, lr_scheduler=scheduler2, device=device)
fit2 = trainer2.fit(train_dl, validate_dl, num_epochs=10, checkpoints=checkpoints, early_stopping=4)


my_loss1 = Loss(lam_w=1.0)
# optimizer
for p in net.layer1.parameters():
    p.requires_grad = True
for p in net.layer2.parameters():
    p.requires_grad = False
print("layer 1 trainable:", count_trainable(net))
optimizer1 = torch.optim.AdamW([p for p in net.layer1.parameters() if p.requires_grad], lr=1e-5, weight_decay=1e-5)
# scheduler
scheduler1 = ReduceLROnPlateau(optimizer1, factor=0.1, patience=1, threshold=0.004, min_lr=1e-6)
# training
trainer1 = TorchTrainer2(net, my_loss1, optimizer1, lr_scheduler=scheduler1, device=device)
fit1 = trainer1.fit(train_dl, validate_dl, num_epochs=2, checkpoints=checkpoints, early_stopping=4)


# loss
my_loss12 = Loss(lam_w=2.0)
# optimizer
for p in net.layer1.parameters():
    p.requires_grad = True
for p in net.layer2.parameters():
    p.requires_grad = True
print("layer 1 and 2 trainable:", count_trainable(net))
optimizer12 = torch.optim.AdamW(
    [
        {"params": net.parameters(), "lr": 1e-5, "weight_decay": 1e-4},
    ]
)

scheduler12 = ReduceLROnPlateau(optimizer12, factor=0.1, patience=1, threshold=0.004, min_lr=1e-6)
trainer12 = TorchTrainer2(net, my_loss12, optimizer12, lr_scheduler=scheduler12, device=device)
fit12 = trainer12.fit(train_dl, validate_dl, num_epochs=2, checkpoints=checkpoints, early_stopping=4)



FileNotFoundError: [WinError 3] The system cannot find the path specified: '/zjbd/zd1/dafei/piv_training/jet320\\y'